# 02 — Behavioral EDA & Hypothesis Generation

The first EDA question was straightforward:

> What early repayment behavior is associated with eventual default?

The exploration starts with missed installments, then separates persistence, recovery and relative/normalized measures.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/synthetic/synthetic_early_modeling_base.csv")
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

## Target

In [ ]:
target = (
    df["is_good_or_bad"]
    .value_counts()
    .rename(index={0: "Good", 1: "Default"})
    .to_frame("count")
)

target["rate"] = target["count"] / len(df)
target

## Frequency

In [ ]:
frequency_default = (
    df.groupby("frequency_name", observed=True)["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .sort_values("default_rate", ascending=False)
)

frequency_default["default_rate_pct"] = (
    frequency_default["default_rate"] * 100
).round(2)

frequency_default

In [ ]:
frequency_default["default_rate_pct"].plot(
    kind="bar", figsize=(7, 4)
)
plt.ylabel("Eventual default rate (%)")
plt.xlabel("Repayment frequency")
plt.title("Default rate by repayment frequency")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Missed installments

In [ ]:
missed_rate = (
    df.groupby("early_missed_installment_count")["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

missed_rate["default_rate_pct"] = missed_rate["default_rate"] * 100
missed_rate.head(15)

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(
    missed_rate["early_missed_installment_count"],
    missed_rate["default_rate_pct"],
    marker="o"
)
plt.xlabel("Early missed-installment count")
plt.ylabel("Eventual default rate (%)")
plt.title("Default rate by early missed-installment count")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Observation

Missed-installment count is a natural candidate signal. The next question is whether **the pattern of misses** matters, not only the count.

## Consecutive misses

In [ ]:
df["has_consecutive_miss"] = (
    df["early_max_consecutive_missed"] >= 2
).astype(int)

consecutive_rate = (
    df.groupby("has_consecutive_miss")["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
)

consecutive_rate["default_rate_pct"] = consecutive_rate["default_rate"] * 100
consecutive_rate

### Hypothesis H1

Loans with persistent consecutive misses may have higher eventual-default risk than loans with isolated misses.

## Recovery timing

In [ ]:
recovery_rate = (
    df.groupby("early_recovery_delay_cycles")["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

recovery_rate["default_rate_pct"] = recovery_rate["default_rate"] * 100
recovery_rate

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(
    recovery_rate["early_recovery_delay_cycles"],
    recovery_rate["default_rate_pct"],
    marker="o"
)
plt.xlabel("Recovery-delay cycles")
plt.ylabel("Eventual default rate (%)")
plt.title("Default rate by recovery delay")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Hypothesis H2

Slower recovery after missed payments may be associated with higher eventual-default risk.

## Normalize overdue duration

In [ ]:
frequency_days = {
    "Weekly": 7,
    "Bi-weekly": 14,
    "Monthly": 28,
}

df["pass_due_cycle_ratio"] = (
    df["early_max_overdue_days"]
    / df["frequency_name"].map(frequency_days)
)

df[[
    "frequency_name",
    "early_max_overdue_days",
    "pass_due_cycle_ratio"
]].head(10)

In [ ]:
bins = [-0.001, 0, 0.5, 1, 1.5, 2, 3, np.inf]
labels = ["0", "0–0.5", "0.5–1", "1–1.5", "1.5–2", "2–3", ">3"]

df["pass_due_cycle_bin"] = pd.cut(
    df["pass_due_cycle_ratio"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

cycle_rate = (
    df.groupby("pass_due_cycle_bin", observed=False)["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

cycle_rate["default_rate_pct"] = cycle_rate["default_rate"] * 100
cycle_rate

### Hypothesis H3

Cycle-normalized overdue persistence may be more comparable across different repayment schedules than raw overdue days.

## Relative overdue burden

In [ ]:
df["overdue_proportion"] = df["overdue_proportion"].clip(0, 1)

overdue_bins = pd.qcut(
    df["overdue_proportion"],
    q=10,
    duplicates="drop"
)

overdue_rate = (
    df.groupby(overdue_bins, observed=True)["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

overdue_rate["default_rate_pct"] = overdue_rate["default_rate"] * 100
overdue_rate

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(
    range(len(overdue_rate)),
    overdue_rate["default_rate_pct"],
    marker="o"
)
plt.xlabel("Overdue-proportion decile")
plt.ylabel("Eventual default rate (%)")
plt.title("Default rate across overdue-proportion deciles")
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### Hypothesis H4

A higher relative overdue burden may be associated with higher eventual-default risk.

## Missed-installment proportion

In [ ]:
missed_prop_bins = pd.qcut(
    df["missed_installment_proportion"],
    q=8,
    duplicates="drop"
)

missed_prop_summary = (
    df.groupby(missed_prop_bins, observed=True)["is_good_or_bad"]
      .agg(["count", "mean"])
      .rename(columns={"mean": "default_rate"})
      .reset_index()
)

missed_prop_summary["default_rate_pct"] = (
    missed_prop_summary["default_rate"] * 100
)

missed_prop_summary

### Hypothesis H5

The proportion of missed installments may contain information beyond the raw count.

## Takeaways

The exploratory stage produces candidate signals for formal testing:

- missed-installment count
- consecutive misses
- recovery delay
- normalized overdue persistence
- overdue proportion
- missed-installment proportion
- repayment frequency

These are hypotheses, not final model features.